In [0]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

np.random.seed(42)

print("Spark version:", spark.version)
print("Databricks Spark session initialized successfully.")

Spark version: 4.2.0
Databricks Spark session initialized successfully.


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.fintech_pipeline
""")

spark.sql("USE workspace.fintech_pipeline")

print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current schema:", spark.sql("SELECT current_schema()").first()[0])

Current catalog: workspace
Current schema: fintech_pipeline


In [0]:
num_records = 100_000

currencies = ["USD", "INR", "EUR", "GBP"]
statuses = ["SUCCESS", "FAILED", "PENDING"]

data = {
    "transaction_id": [f"TXN{i:08d}" for i in range(1, num_records + 1)],
    "customer_id": np.random.randint(1000, 10000, num_records),
    "amount": np.round(np.random.uniform(10, 10000, num_records), 2),
    "currency": np.random.choice(currencies, num_records),
    "transaction_status": np.random.choice(
        statuses, num_records, p=[0.85, 0.10, 0.05]
    ),
    "transaction_timestamp": [
        datetime(2026, 1, 1) + timedelta(minutes=int(x))
        for x in np.random.randint(0, 43200, num_records)
    ]
}

transactions_df = pd.DataFrame(data)

raw_df = spark.createDataFrame(transactions_df)

raw_df.write.format("delta").mode("overwrite").saveAsTable(
    "raw_transactions_source"
)

raw_df = spark.table("raw_transactions_source")

print("Synthetic transaction data generated successfully!")
print("Total records:", raw_df.count())

display(raw_df.limit(5))

Synthetic transaction data generated successfully!
Total records: 100000


transaction_id,customer_id,amount,currency,transaction_status,transaction_timestamp
TXN00025001,5926,7465.26,USD,SUCCESS,2026-01-28T06:47:00.000Z
TXN00025002,2055,4016.34,USD,SUCCESS,2026-01-10T08:21:00.000Z
TXN00025003,2254,8989.35,USD,SUCCESS,2026-01-19T15:01:00.000Z
TXN00025004,5286,1046.0,EUR,PENDING,2026-01-09T23:31:00.000Z
TXN00025005,9009,229.36,GBP,SUCCESS,2026-01-23T02:14:00.000Z


In [0]:
from pyspark.sql.functions import col, lit

ordered_df = raw_df.orderBy("transaction_id")

duplicate_records = ordered_df.limit(2000)

missing_customer = (
    ordered_df.limit(1000)
    .withColumn("customer_id", lit(None).cast("integer"))
)

missing_amount = (
    ordered_df.limit(1000)
    .withColumn("amount", lit(None).cast("double"))
)

invalid_status = (
    ordered_df.limit(1000)
    .withColumn("transaction_status", lit("UNKNOWN"))
)

negative_amount = (
    ordered_df.limit(1000)
    .withColumn("amount", -col("amount"))
)

missing_currency = (
    ordered_df.limit(1000)
    .withColumn("currency", lit(None).cast("string"))
)

source_with_issues = (
    raw_df
    .unionByName(duplicate_records)
    .unionByName(missing_customer)
    .unionByName(missing_amount)
    .unionByName(invalid_status)
    .unionByName(negative_amount)
    .unionByName(missing_currency)
)

source_with_issues.write.format("delta").mode("overwrite").saveAsTable(
    "incoming_transactions"
)

print("Original records:", raw_df.count())
print("Records after introducing issues:", source_with_issues.count())
print("Corrupted source dataset saved successfully!")

Original records: 100000
Records after introducing issues: 107000
Corrupted source dataset saved successfully!


In [0]:
incoming_df = spark.table("incoming_transactions")

incoming_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_transactions")

bronze_df = spark.table("bronze_transactions")

print("Bronze layer created successfully!")
print("Bronze record count:", bronze_df.count())

bronze_df.printSchema()

Bronze layer created successfully!
Bronze record count: 107000
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)



In [0]:
bronze_df.select(
    F.count("*").alias("total_records"),

    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0))
        .alias("missing_customer_id"),

    F.sum(F.when(F.col("amount").isNull(), 1).otherwise(0))
        .alias("missing_amount"),

    F.sum(F.when(F.col("currency").isNull(), 1).otherwise(0))
        .alias("missing_currency"),

    F.sum(F.when(
        ~F.col("transaction_status").isin(
            "SUCCESS", "FAILED", "PENDING"
        ), 1
    ).otherwise(0)).alias("invalid_status"),

    F.sum(F.when(F.col("amount") < 0, 1).otherwise(0))
        .alias("negative_amount"),

    F.count("transaction_id").alias("non_null_transaction_ids")
).show(truncate=False)

+-------------+-------------------+--------------+----------------+--------------+---------------+------------------------+
|total_records|missing_customer_id|missing_amount|missing_currency|invalid_status|negative_amount|non_null_transaction_ids|
+-------------+-------------------+--------------+----------------+--------------+---------------+------------------------+
|107000       |1000               |1000          |1000            |1000          |1000           |107000                  |
+-------------+-------------------+--------------+----------------+--------------+---------------+------------------------+



In [0]:
duplicate_transactions = (
    bronze_df
    .groupBy("transaction_id")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
)

print("Duplicate transaction IDs:", duplicate_transactions.count())

display(duplicate_transactions.limit(10))

Duplicate transaction IDs: 2000


transaction_id,record_count
TXN00000001,7
TXN00000002,7
TXN00000003,7
TXN00000004,7
TXN00000005,7
TXN00000006,7
TXN00000007,7
TXN00000008,7
TXN00000009,7
TXN00000010,7


In [0]:
validated_df = bronze_df.withColumn(
    "rejection_reasons",
    F.expr("""
        filter(
            array(
                CASE WHEN transaction_id IS NULL
                    THEN 'MISSING_TRANSACTION_ID' END,
                CASE WHEN customer_id IS NULL
                    THEN 'MISSING_CUSTOMER_ID' END,
                CASE WHEN amount IS NULL
                    THEN 'MISSING_AMOUNT' END,
                CASE WHEN amount <= 0
                    THEN 'INVALID_AMOUNT' END,
                CASE WHEN currency IS NULL
                    THEN 'MISSING_CURRENCY' END,
                CASE WHEN currency NOT IN ('USD', 'INR', 'EUR', 'GBP')
                    THEN 'INVALID_CURRENCY' END,
                CASE WHEN transaction_status IS NULL
                    THEN 'MISSING_STATUS' END,
                CASE WHEN transaction_status NOT IN ('SUCCESS', 'FAILED', 'PENDING')
                    THEN 'INVALID_STATUS' END,
                CASE WHEN transaction_timestamp IS NULL
                    THEN 'MISSING_TIMESTAMP' END
            ),
            x -> x IS NOT NULL
        )
    """)
)

invalid_df = validated_df.filter(F.size("rejection_reasons") > 0)
valid_df = validated_df.filter(F.size("rejection_reasons") == 0)

print("Total Bronze records:", bronze_df.count())
print("Invalid records:", invalid_df.count())
print("Valid records:", valid_df.count())

display(
    invalid_df.select("transaction_id", "rejection_reasons").limit(10)
)

Total Bronze records: 107000
Invalid records: 5000
Valid records: 102000


transaction_id,rejection_reasons
TXN00000001,List(MISSING_CUSTOMER_ID)
TXN00000002,List(MISSING_CUSTOMER_ID)
TXN00000003,List(MISSING_CUSTOMER_ID)
TXN00000004,List(MISSING_CUSTOMER_ID)
TXN00000005,List(MISSING_CUSTOMER_ID)
TXN00000006,List(MISSING_CUSTOMER_ID)
TXN00000007,List(MISSING_CUSTOMER_ID)
TXN00000008,List(MISSING_CUSTOMER_ID)
TXN00000009,List(MISSING_CUSTOMER_ID)
TXN00000010,List(MISSING_CUSTOMER_ID)


In [0]:
invalid_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("quarantine_transactions")

quarantine_df = spark.table("quarantine_transactions")

print("Quarantine table created successfully!")
print("Quarantined records:", quarantine_df.count())

display(
    quarantine_df.select(
        "transaction_id",
        "rejection_reasons"
    ).limit(10)
)

Quarantine table created successfully!
Quarantined records: 5000


transaction_id,rejection_reasons
TXN00000001,List(MISSING_CUSTOMER_ID)
TXN00000002,List(MISSING_CUSTOMER_ID)
TXN00000003,List(MISSING_CUSTOMER_ID)
TXN00000004,List(MISSING_CUSTOMER_ID)
TXN00000005,List(MISSING_CUSTOMER_ID)
TXN00000006,List(MISSING_CUSTOMER_ID)
TXN00000007,List(MISSING_CUSTOMER_ID)
TXN00000008,List(MISSING_CUSTOMER_ID)
TXN00000009,List(MISSING_CUSTOMER_ID)
TXN00000010,List(MISSING_CUSTOMER_ID)


In [0]:
bronze_count = bronze_df.count()
valid_count = valid_df.count()
quarantine_count = quarantine_df.count()

assert bronze_count == valid_count + quarantine_count, \
    "Record distribution check failed!"

print("Bronze records:", bronze_count)
print("Valid records:", valid_count)
print("Quarantined records:", quarantine_count)
print("Record distribution check: PASSED")

Bronze records: 107000
Valid records: 102000
Quarantined records: 5000
Record distribution check: PASSED


In [0]:
duplicate_summary = (
    valid_df
    .groupBy("transaction_id")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
)

total_duplicate_ids = duplicate_summary.count()

excess_duplicate_rows = (
    duplicate_summary
    .select(F.sum(F.col("record_count") - 1).alias("excess_rows"))
    .first()["excess_rows"]
)

dedup_window = Window.partitionBy("transaction_id").orderBy(
    F.col("transaction_timestamp").desc(),
    F.col("amount").desc_nulls_last(),
    F.col("customer_id").desc_nulls_last()
)

silver_df = (
    valid_df
    .withColumn("row_num", F.row_number().over(dedup_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "rejection_reasons")
)

print("Duplicate transaction IDs:", total_duplicate_ids)
print("Excess duplicate rows:", excess_duplicate_rows)
print("Records before deduplication:", valid_df.count())
print("Records after deduplication:", silver_df.count())

remaining_duplicates = (
    silver_df.groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Remaining duplicate IDs:", remaining_duplicates)

assert remaining_duplicates == 0
assert valid_df.count() == silver_df.count() + excess_duplicate_rows

print("Silver deduplication integrity check: PASSED")

Duplicate transaction IDs: 2000
Excess duplicate rows: 2000
Records before deduplication: 102000
Records after deduplication: 100000
Remaining duplicate IDs: 0
Silver deduplication integrity check: PASSED


In [0]:
silver_quality = silver_df.select(
    F.count("*").alias("total_records"),
    F.countDistinct("transaction_id").alias("unique_transactions"),

    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0))
        .alias("missing_customer_ids"),

    F.sum(F.when(
        F.col("amount").isNull() | (F.col("amount") <= 0), 1
    ).otherwise(0)).alias("invalid_amounts"),

    F.sum(F.when(F.col("currency").isNull(), 1).otherwise(0))
        .alias("missing_currencies"),

    F.sum(F.when(
        ~F.col("transaction_status").isin(
            "SUCCESS", "FAILED", "PENDING"
        ), 1
    ).otherwise(0)).alias("invalid_statuses")
).first().asDict()

assert silver_quality["total_records"] == silver_quality["unique_transactions"]
assert silver_quality["missing_customer_ids"] == 0
assert silver_quality["invalid_amounts"] == 0
assert silver_quality["missing_currencies"] == 0
assert silver_quality["invalid_statuses"] == 0

print("Silver quality metrics:")
print(silver_quality)
print("All Silver quality assertions passed!")

Silver quality metrics:
{'total_records': 100000, 'unique_transactions': 100000, 'missing_customer_ids': 0, 'invalid_amounts': 0, 'missing_currencies': 0, 'invalid_statuses': 0}
All Silver quality assertions passed!


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_transactions")

silver_table = spark.table("silver_transactions")

print("Silver Delta table created successfully!")
print("Silver record count:", silver_table.count())

Silver Delta table created successfully!
Silver record count: 100000


In [0]:
gold_df = (
    silver_table
    .withColumn(
        "transaction_date",
        F.to_date("transaction_timestamp")
    )
    .groupBy(
        "transaction_date",
        "currency",
        "transaction_status"
    )
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("average_amount")
    )
)

print("Gold transformation completed!")
print("Gold records:", gold_df.count())

display(
    gold_df.orderBy(
        "transaction_date",
        "currency",
        "transaction_status"
    ).limit(10)
)

Gold transformation completed!
Gold records: 360


transaction_date,currency,transaction_status,total_transactions,total_amount,average_amount
2026-01-01,EUR,FAILED,93,467253.7,5024.23
2026-01-01,EUR,PENDING,49,278031.25,5674.11
2026-01-01,EUR,SUCCESS,688,3458675.82,5027.15
2026-01-01,GBP,FAILED,84,382741.37,4556.44
2026-01-01,GBP,PENDING,39,194716.56,4992.73
2026-01-01,GBP,SUCCESS,717,3622938.54,5052.91
2026-01-01,INR,FAILED,82,437186.77,5331.55
2026-01-01,INR,PENDING,49,254555.15,5195.0
2026-01-01,INR,SUCCESS,727,3659528.56,5033.74
2026-01-01,USD,FAILED,85,440184.99,5178.65


In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_transaction_summary")

gold_table = spark.table("gold_daily_transaction_summary")

print("Gold Delta table created successfully!")
print("Gold record count:", gold_table.count())

Gold Delta table created successfully!
Gold record count: 360


In [0]:
silver_total = silver_table.agg(
    F.sum("amount").alias("total")
).first()["total"]

gold_total = gold_table.agg(
    F.sum("total_amount").alias("total")
).first()["total"]

reconciliation_difference = silver_total - gold_total
absolute_reconciliation_difference = abs(reconciliation_difference)

tolerance = 0.01

reconciliation_status = (
    "PASSED"
    if absolute_reconciliation_difference <= tolerance
    else "FAILED"
)

print("Silver total amount:", round(silver_total, 2))
print("Gold total amount:", round(gold_total, 2))
print("Difference:", round(reconciliation_difference, 2))
print("Reconciliation tolerance:", tolerance)
print("Reconciliation status:", reconciliation_status)

assert reconciliation_status == "PASSED", \
    "Financial reconciliation failed!"

print("Financial reconciliation assertion: PASSED")

Silver total amount: 501449258.07
Gold total amount: 501449258.07
Difference: 0.0
Reconciliation tolerance: 0.01
Reconciliation status: PASSED
Financial reconciliation assertion: PASSED


In [0]:
gold_duplicate_keys = (
    gold_table
    .groupBy(
        "transaction_date",
        "currency",
        "transaction_status"
    )
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
)

gold_quality = gold_table.select(
    F.count("*").alias("total_records"),

    F.sum(F.when(F.col("transaction_date").isNull(), 1).otherwise(0))
        .alias("null_transaction_dates"),

    F.sum(F.when(F.col("total_transactions") <= 0, 1).otherwise(0))
        .alias("invalid_transaction_counts"),

    F.sum(F.when(F.col("total_amount") <= 0, 1).otherwise(0))
        .alias("invalid_total_amounts")
).first().asDict()

assert gold_duplicate_keys.count() == 0
assert gold_quality["null_transaction_dates"] == 0
assert gold_quality["invalid_transaction_counts"] == 0
assert gold_quality["invalid_total_amounts"] == 0

print("Gold quality metrics:")
print(gold_quality)
print("Duplicate Gold keys:", gold_duplicate_keys.count())
print("All Gold quality assertions passed!")

Gold quality metrics:
{'total_records': 360, 'null_transaction_dates': 0, 'invalid_transaction_counts': 0, 'invalid_total_amounts': 0}
Duplicate Gold keys: 0
All Gold quality assertions passed!


In [0]:
pipeline_status = (
    "PASSED"
    if (
        reconciliation_status == "PASSED"
        and silver_quality["total_records"] == silver_quality["unique_transactions"]
        and silver_quality["missing_customer_ids"] == 0
        and silver_quality["invalid_amounts"] == 0
        and silver_quality["missing_currencies"] == 0
        and silver_quality["invalid_statuses"] == 0
        and gold_duplicate_keys.count() == 0
        and gold_quality["null_transaction_dates"] == 0
        and gold_quality["invalid_transaction_counts"] == 0
        and gold_quality["invalid_total_amounts"] == 0
    )
    else "FAILED"
)

pipeline_report = {
    "bronze_records": bronze_df.count(),
    "quarantined_records": quarantine_df.count(),
    "silver_records": silver_table.count(),
    "gold_records": gold_table.count(),
    "pipeline_status": pipeline_status,
    "silver_quality_status": "PASSED",
    "gold_quality_status": "PASSED",
    "reconciliation_status": reconciliation_status,
    "silver_duplicate_ids": (
        silver_table.count()
        - silver_table.select("transaction_id").distinct().count()
    ),
    "gold_duplicate_keys": gold_duplicate_keys.count(),
    "silver_total_amount": round(silver_total, 2),
    "gold_total_amount": round(gold_total, 2),
    "reconciliation_difference": round(reconciliation_difference, 2),
    "absolute_reconciliation_difference": round(
        absolute_reconciliation_difference, 2
    ),
    "reconciliation_tolerance": tolerance
}

print(json.dumps(pipeline_report, indent=4))

assert pipeline_status == "PASSED", "Pipeline validation failed!"

print("\nOverall pipeline execution:", pipeline_status)

{
    "bronze_records": 107000,
    "quarantined_records": 5000,
    "silver_records": 100000,
    "gold_records": 360,
    "pipeline_status": "PASSED",
    "silver_quality_status": "PASSED",
    "gold_quality_status": "PASSED",
    "reconciliation_status": "PASSED",
    "silver_duplicate_ids": 0,
    "gold_duplicate_keys": 0,
    "silver_total_amount": 501449258.07,
    "gold_total_amount": 501449258.07,
    "reconciliation_difference": 0.0,
    "absolute_reconciliation_difference": 0.0,
    "reconciliation_tolerance": 0.01
}

Overall pipeline execution: PASSED


In [0]:
spark.sql("SHOW TABLES IN workspace.fintech_pipeline").show(
    truncate=False
)

+----------------+------------------------------+-----------+
|database        |tableName                     |isTemporary|
+----------------+------------------------------+-----------+
|fintech_pipeline|bronze_transactions           |false      |
|fintech_pipeline|gold_daily_transaction_summary|false      |
|fintech_pipeline|incoming_transactions         |false      |
|fintech_pipeline|quarantine_transactions       |false      |
|fintech_pipeline|raw_transactions_source       |false      |
|fintech_pipeline|silver_transactions           |false      |
+----------------+------------------------------+-----------+



In [0]:
report_df = spark.createDataFrame([pipeline_report])

report_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_validation_report")

display(spark.table("pipeline_validation_report"))

absolute_reconciliation_difference,bronze_records,gold_duplicate_keys,gold_quality_status,gold_records,gold_total_amount,pipeline_status,quarantined_records,reconciliation_difference,reconciliation_status,reconciliation_tolerance,silver_duplicate_ids,silver_quality_status,silver_records,silver_total_amount
0.0,107000,0,PASSED,360,5.0144925807E8,PASSED,5000,0.0,PASSED,0.01,0,PASSED,100000,5.0144925807E8
